# AIST-FYP Colab Wikipedia Preprocessing

This notebook preprocesses Wikipedia corpus data on Colab by reusing the project scripts:
- `scripts/download_wikipedia.py`
- `scripts/prepare_wikipedia_chunks.py`
- `scripts/generate_embeddings.py`
- `scripts/build_faiss_index.py`
- `scripts/build_bm25_index.py`

Default mode is **production** with **FAISS + BM25** and uses a **Google Drive persistent work root** so checkpoints survive Colab runtime resets.

## 🔑 Optional Secrets

If needed, add secrets in Colab sidebar (🔑):
- `HUGGINGFACE_TOKEN` (optional)

This preprocessing flow does not require OpenAI/DeepSeek keys.

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["preprocessing"]

# Processing strategy
STRATEGY = "production"  # development | validation | production
DUMP_DATE = "latest"     # only used for production
MAX_ARTICLES_OVERRIDE = None  # for dev/validation quick runs, e.g., 5000

# Runtime/storage policy
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/AIST-FYP-colab-preprocess"
LOCAL_WORK_ROOT = f"{DRIVE_OUTPUT_ROOT}/work"  # Persistent workspace to survive Colab runtime resets
RUN_TAG = "wiki_preprocess_production"
REUSE_EXISTING_ARTICLE_JSONL = True  # production: skip source download if intermediate exists and reset is off
SKIP_EXPORT_COPY_WHEN_PERSISTENT = True  # if LOCAL_WORK_ROOT is on Drive, avoid duplicate artifact copy in Step 6

# Build targets
BUILD_FAISS = True
BUILD_BM25 = True

# FAISS settings
FAISS_INDEX_TYPE = "IVFFLAT"  # FLAT | IVFFLAT | HNSW
FAISS_NLIST = 4096
FAISS_NPROBE = 128
FAISS_HNSW_M = 32

# Checkpoint behavior
RESUME = True
RESET_CHECKPOINT = False
CHUNKING_CHECKPOINT_INTERVAL = 1000
EMBEDDING_CHECKPOINT_INTERVAL = 10000
FAISS_CHECKPOINT_INTERVAL = 200000
FAISS_ADD_BATCH_SIZE = 50000
BM25_CHECKPOINT_INTERVAL = 5000

# Performance knobs
EMBED_BATCH_SIZE_OVERRIDE = None
DISABLE_FP16 = False
INSTALL_SPACY_MODEL = True

In [ ]:
import os
import json
import shlex
import shutil
import subprocess
import yaml
from datetime import datetime
from pathlib import Path

def shell_join(parts):
    return " ".join(shlex.quote(str(p)) for p in parts)

def run(cmd, cwd=None, check=True, stream=True):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.exists() or link_path.is_symlink():
        if link_path.is_symlink() and link_path.resolve() == target_path.resolve():
            return
        if link_path.is_symlink() or link_path.is_file():
            link_path.unlink()
        else:
            shutil.rmtree(link_path)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(target_path, target_is_directory=True)
    return link_path, target_path, "linked"

In [ ]:
# Mount Drive + clone/update repo + install dependencies
from google.colab import drive
drive.mount('/content/drive')

if not str(LOCAL_WORK_ROOT).startswith('/content/drive/'):
    print(f"⚠️ LOCAL_WORK_ROOT is not on Drive: {LOCAL_WORK_ROOT}")
    print("Checkpoint persistence across runtime resets may be lost.")

ensure_dir(LOCAL_WORK_ROOT)
print("Persistent LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)
print("Drive output root:", DRIVE_OUTPUT_ROOT)

repo_path = Path(REPO_DIR)
if repo_path.exists():
    print(f"Repo exists: {repo_path}")
    run("git fetch --all", cwd=REPO_DIR)
    run(f"git checkout {REPO_BRANCH}", cwd=REPO_DIR)
    run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR, check=False)
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

run("python -m pip install -U pip wheel setuptools", stream=True)
run("python -m pip install -U uv", stream=True)

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False, stream=True)

spacy_model_wheel = "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
spacy_install_cmd = "python -m spacy download en_core_web_sm"

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    spacy_install_cmd = f"uv pip install --python {uv_python} {spacy_model_wheel}"
    print(f"✅ uv sync complete: {uv_project}")
else:
    print('\n⚠️ uv sync failed. Falling back to pip requirements install...')

    # Colab-safe pip fallback for torch +cu121 pins
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\n⚠️ Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')

        run('python - <<"PY"\nimport torch\nimport torchvision\nimport torchaudio\nprint("torch", torch.__version__)\nprint("torchvision", torchvision.__version__)\nprint("torchaudio", torchaudio.__version__)\nPY', cwd=REPO_DIR)
        run(f"pip install -r {temp_req}", cwd=REPO_DIR, stream=True)

if INSTALL_SPACY_MODEL:
    run(spacy_install_cmd, cwd=REPO_DIR, stream=True)

# Quick CLI sanity checks
run("python scripts/download_wikipedia.py --help", cwd=REPO_DIR)
run("python scripts/prepare_wikipedia_chunks.py --help", cwd=REPO_DIR)
run("python scripts/generate_embeddings.py --help", cwd=REPO_DIR)
run("python scripts/build_faiss_index.py --help", cwd=REPO_DIR)
run("python scripts/build_bm25_index.py --help", cwd=REPO_DIR)

In [ ]:
# Fail fast if no GPU
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab: Runtime -> Change runtime type -> GPU")

print("GPU:", torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())

In [ ]:
# Build colab preprocessing config with persistent LOCAL_WORK_ROOT paths
base_config_path = Path(REPO_DIR) / "config.yaml"
colab_config_path = Path(REPO_DIR) / "config.colab.preprocess.yaml"

with open(base_config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

local_data = Path(LOCAL_WORK_ROOT) / "data"
cfg.setdefault("data", {})
cfg["data"]["wikipedia_dump"] = str(local_data / "raw" / "enwiki-latest-pages-articles.xml.bz2")
cfg["data"]["wikipedia_sample_dev"] = str(local_data / "raw" / "wiki_sample_development.jsonl")
cfg["data"]["wikipedia_sample_val"] = str(local_data / "raw" / "wiki_sample_validation.jsonl")
cfg["data"]["processed_chunks"] = str(local_data / "processed" / "wiki_chunks_{strategy}.jsonl")
cfg["data"]["embeddings"] = str(local_data / "embeddings" / "wiki_embeddings_{strategy}.npy")
cfg["data"]["embeddings_metadata"] = str(local_data / "embeddings" / "metadata_{strategy}.json")
cfg["data"]["faiss_index"] = str(local_data / "indexes" / "{strategy}" / "faiss.index")
cfg["data"]["index_metadata"] = str(local_data / "indexes" / "{strategy}" / "metadata.pkl")
cfg["data"]["index_config"] = str(local_data / "indexes" / "{strategy}" / "index_config.json")
cfg["data"]["bm25_index"] = str(local_data / "indexes" / "{strategy}" / "bm25_index.pkl")

cfg.setdefault("processing", {})["device"] = "cuda"
cfg.setdefault("verification", {}).setdefault("nli", {})["device"] = "cuda"
cfg.setdefault("verification", {}).setdefault("self_agreement", {})["device"] = "cuda"

cfg.setdefault("checkpointing", {})
cfg["checkpointing"]["checkpoint_dir"] = str(local_data / "embeddings" / "checkpoints")
cfg["checkpointing"].setdefault("chunking", {})["checkpoint_dir"] = str(local_data / "checkpoints" / "chunking")
cfg["checkpointing"].setdefault("faiss", {})["checkpoint_dir"] = str(local_data / "checkpoints" / "faiss")
cfg["checkpointing"].setdefault("bm25", {})["checkpoint_dir"] = str(local_data / "checkpoints" / "bm25")

ensure_dir(local_data / "raw")
ensure_dir(local_data / "processed")
ensure_dir(local_data / "embeddings")
ensure_dir(local_data / "indexes")
ensure_dir(local_data / "checkpoints")

with open(colab_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Wrote config:", colab_config_path)
print("Local work root:", LOCAL_WORK_ROOT)
print("Strategy:", STRATEGY)
print("Chunk checkpoint dir:", cfg["checkpointing"]["chunking"]["checkpoint_dir"])
print("FAISS checkpoint dir:", cfg["checkpointing"]["faiss"]["checkpoint_dir"])
print("BM25 checkpoint dir:", cfg["checkpointing"]["bm25"]["checkpoint_dir"])
print("Embedding checkpoint dir:", cfg["checkpointing"]["checkpoint_dir"])

In [ ]:
# Step 1: Download Wikipedia source data (optional if intermediate article JSONL exists)
article_jsonl = str(Path(LOCAL_WORK_ROOT) / "data" / "processed" / f"wiki_articles_{STRATEGY}.jsonl")
article_jsonl_path = Path(article_jsonl)
chunk_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "chunking"

print(f"Step 1 checkpoint context: chunking checkpoints at {chunk_ckpt_dir}")

should_download_source = True
if STRATEGY == "production" and REUSE_EXISTING_ARTICLE_JSONL and article_jsonl_path.exists() and not RESET_CHECKPOINT:
    should_download_source = False
    print(f"Skipping Step 1 download: reusing intermediate article JSONL at {article_jsonl_path}")

if should_download_source:
    download_cmd = [
        "python",
        "scripts/download_wikipedia.py",
        "--strategy", STRATEGY,
        "--config", "config.colab.preprocess.yaml",
    ]

    if STRATEGY == "production":
        download_cmd += ["--dump-date", DUMP_DATE]
    elif MAX_ARTICLES_OVERRIDE is not None:
        download_cmd += ["--max-articles", str(MAX_ARTICLES_OVERRIDE)]

    run(shell_join(download_cmd), cwd=REPO_DIR, stream=True)

In [ ]:
# Step 2: Prepare sentence-level chunks
article_jsonl = str(Path(LOCAL_WORK_ROOT) / "data" / "processed" / f"wiki_articles_{STRATEGY}.jsonl")
chunk_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "chunking"
print(f"Step 2 checkpoint dir: {chunk_ckpt_dir}")

chunk_cmd = [
    "python",
    "scripts/prepare_wikipedia_chunks.py",
    "--strategy", STRATEGY,
    "--config", "config.colab.preprocess.yaml",
    "--article-jsonl", article_jsonl,
    "--checkpoint-interval", str(CHUNKING_CHECKPOINT_INTERVAL),
]

if RESUME:
    chunk_cmd.append("--resume")
else:
    chunk_cmd.append("--no-resume")

if RESET_CHECKPOINT:
    chunk_cmd.append("--reset-checkpoint")

run(shell_join(chunk_cmd), cwd=REPO_DIR, stream=True)

In [ ]:
# Step 3: Generate embeddings
embedding_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "embeddings" / "checkpoints"
print(f"Step 3 checkpoint dir: {embedding_ckpt_dir}")

embed_cmd = [
    "python",
    "scripts/generate_embeddings.py",
    "--strategy", STRATEGY,
    "--config", "config.colab.preprocess.yaml",
    "--device", "cuda",
    "--checkpoint-interval", str(EMBEDDING_CHECKPOINT_INTERVAL),
]

if EMBED_BATCH_SIZE_OVERRIDE is not None:
    embed_cmd += ["--batch-size", str(EMBED_BATCH_SIZE_OVERRIDE)]

if DISABLE_FP16:
    embed_cmd.append("--no-fp16")

run(shell_join(embed_cmd), cwd=REPO_DIR, stream=True)

In [ ]:
# Step 4: Build FAISS index
faiss_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "faiss"
print(f"Step 4 checkpoint dir: {faiss_ckpt_dir}")

if BUILD_FAISS:
    faiss_cmd = [
        "python",
        "scripts/build_faiss_index.py",
        "--strategy", STRATEGY,
        "--config", "config.colab.preprocess.yaml",
        "--index-type", FAISS_INDEX_TYPE,
        "--nlist", str(FAISS_NLIST),
        "--nprobe", str(FAISS_NPROBE),
        "--hnsw-m", str(FAISS_HNSW_M),
        "--checkpoint-interval", str(FAISS_CHECKPOINT_INTERVAL),
        "--add-batch-size", str(FAISS_ADD_BATCH_SIZE),
    ]

    if RESUME:
        faiss_cmd.append("--resume")
    else:
        faiss_cmd.append("--no-resume")

    if RESET_CHECKPOINT:
        faiss_cmd.append("--reset-checkpoint")

    run(shell_join(faiss_cmd), cwd=REPO_DIR, stream=True)
else:
    print("BUILD_FAISS=False, skipped.")

In [ ]:
# Step 5: Build BM25 index
bm25_ckpt_dir = Path(LOCAL_WORK_ROOT) / "data" / "checkpoints" / "bm25"
print(f"Step 5 checkpoint dir: {bm25_ckpt_dir}")

if BUILD_BM25:
    bm25_cmd = [
        "python",
        "scripts/build_bm25_index.py",
        "--strategy", STRATEGY,
        "--config", "config.colab.preprocess.yaml",
        "--checkpoint-interval", str(BM25_CHECKPOINT_INTERVAL),
    ]

    if RESUME:
        bm25_cmd.append("--resume")
    else:
        bm25_cmd.append("--no-resume")

    if RESET_CHECKPOINT:
        bm25_cmd.append("--reset-checkpoint")

    run(shell_join(bm25_cmd), cwd=REPO_DIR, stream=True)
else:
    print("BUILD_BM25=False, skipped.")

In [ ]:
# Step 6: Validate artifacts and export manifest/snapshot
with open(Path(REPO_DIR) / "config.colab.preprocess.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

expected = [
    Path(cfg["data"]["processed_chunks"].format(strategy=STRATEGY)),
    Path(cfg["data"]["embeddings"].format(strategy=STRATEGY)),
    Path(cfg["data"]["embeddings_metadata"].format(strategy=STRATEGY)),
    Path(cfg["data"]["faiss_index"].format(strategy=STRATEGY)),
    Path(cfg["data"]["index_metadata"].format(strategy=STRATEGY)),
]

if BUILD_BM25:
    expected.append(Path(cfg["data"]["bm25_index"].format(strategy=STRATEGY)))

missing = [str(p) for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError("Missing expected artifacts:\n" + "\n".join(missing))

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_root = Path(DRIVE_OUTPUT_ROOT) / f"{RUN_TAG}_{STRATEGY}_{run_stamp}"
ensure_dir(export_root)

local_work_root_path = Path(LOCAL_WORK_ROOT).resolve()
artifacts_on_drive = str(local_work_root_path).startswith('/content/drive/')

exported_paths = []
if artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT:
    print("Artifacts are already on Drive; skipping duplicate file copy.")
    print("Writing run manifest only.")
    exported_paths = [str(p) for p in expected]
else:
    for p in expected:
        rel = p.relative_to(local_work_root_path) if str(p).startswith(str(local_work_root_path)) else Path(p.name)
        dest = export_root / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
        exported_paths.append(str(dest))
        print("Exported:", p, "->", dest)

manifest = {
    "timestamp": run_stamp,
    "strategy": STRATEGY,
    "dump_date": DUMP_DATE,
    "build_faiss": BUILD_FAISS,
    "build_bm25": BUILD_BM25,
    "local_work_root": LOCAL_WORK_ROOT,
    "repo_dir": REPO_DIR,
    "export_root": str(export_root),
    "artifacts": [str(p) for p in expected],
    "exported_artifacts": exported_paths,
    "artifacts_already_on_drive": artifacts_on_drive,
    "skipped_copy_when_persistent": bool(artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT),
}

manifest_path = export_root / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("\nDone. Manifest:", manifest_path)